In [7]:
import pandas as pd

# El archivo debe tener los encabezados en la primera fila:
# Nombre | Ciudad | Estado
locations = pd.read_excel(
    'ArchivosExcel/Lugares.xlsx'
)

# Verificar las columnas disponibles
print("Columnas encontradas:", locations.columns.tolist())

if 'Nombre' not in locations.columns:
    raise ValueError("El archivo debe contener una columna llamada 'Nombre'.")

locations.head()


Columnas encontradas: ['País LATAM', 'Nombre', 'Ciudad', 'Estado / Provincia / Departamento']


,País LATAM,Nombre,Ciudad,Estado / Provincia / Departamento
0,MX,IMSS Hospital Centro Medico Siglo XXI,NaN,NaN
1,MX,Ángeles Metropolitano,Ciudad de México,NaN
2,MX,Hospital de alta especialidad del Bajío,NaN,Guanajuato
3,AR,Hospital General de Agudos José María Ramos Mejía,Buenos Aires,NaN
4,CO,Hosítal Universitario San Antonio,Bogotá,NaN


In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import quote_plus
import re
import time

# Iniciar una sesión nueva de Chrome.
options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument('--window-size=1920,1080')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 12)


In [9]:
def clean_value(value):
    # Convierte valores vacíos/NaN a cadena vacía
    if pd.isna(value):
        return ''
    return str(value).strip()


def build_search_query(row):
    # Construye la búsqueda usando únicamente los campos disponibles.
    parts = [
        clean_value(row.get('País LATAM', '')),
        clean_value(row.get('Nombre', '')),
        clean_value(row.get('Ciudad', '')),
        clean_value(row.get('Estado / Provincia / Departamento', ''))
    ]
    return ' '.join(part for part in parts if part)


def get_google_name(driver):
    # Intenta obtener el nombre que muestra la ficha de Google Maps.
    selectors = [
        'h1.DUwDvf',
        'h1[class*="fontHeadlineLarge"]',
        'h1'
    ]

    for selector in selectors:
        try:
            element = WebDriverWait(driver, 3).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector))
            )
            value = element.text.strip()
            if value:
                return value
        except Exception:
            pass

    return 'Name not found'


def get_address_from_maps(driver):
    # Extrae la dirección visible en la ficha del lugar.
    selectors = [
        '[data-item-id="address"]',
        'button[data-item-id="address"]',
        'button[aria-label^="Address:"]',
        'button[aria-label^="Dirección:"]'
    ]

    for selector in selectors:
        try:
            element = WebDriverWait(driver, 4).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector))
            )

            value = element.get_attribute('aria-label') or element.text

            if value:
                value = re.sub(
                    r'^(Address|Dirección):\s*',
                    '',
                    value,
                    flags=re.IGNORECASE
                ).strip()

                if value:
                    return value

        except Exception:
            pass

    return 'Address not found'


def extract_postal_code(address):
    # Extrae un código postal de 5 dígitos de la dirección.
    # Está pensado principalmente para direcciones de México.
    if not address or address == 'Address not found':
        return 'Postal code not found'

    match = re.search(r'(?<!\d)(\d{5})(?:-\d{4})?(?!\d)', address)

    if match:
        return match.group(1)

    return 'Postal code not found'


def click_first_result_if_needed(driver):
    # Si Google Maps devuelve una lista de resultados en vez de abrir una ficha directamente, intenta abrir el primer resultado.
    try:
        results = driver.find_elements(
            By.CSS_SELECTOR,
            'a[href*="/maps/place/"]'
        )

        visible_results = [
            element for element in results
            if element.is_displayed()
        ]

        if visible_results:
            driver.execute_script(
                "arguments[0].click();",
                visible_results[0]
            )
            time.sleep(1.5)
            return True
    except Exception:
        pass

    return False


def search_google_maps(row):
   # Busca un registro de Excel en Google Maps y devuelve nombre, dirección, código postal, query utilizada y estado.
    query = build_search_query(row)

    if not query:
        return {
            'Google_Name': '',
            'Google_Address': '',
            'Postal_Code': '',
            'Search_Query': '',
            'Search_Status': 'EMPTY INPUT'
        }

    print(f'Buscando: {query}')

    try:
        url = (
            'https://www.google.com/maps/search/'
            f'?api=1&query={quote_plus(query)}'
        )

        driver.get(url)

        WebDriverWait(driver, 12).until(
            EC.presence_of_element_located((By.TAG_NAME, 'body'))
        )

        time.sleep(1.2)

        # Si aparece un CAPTCHA o aviso de tráfico inusual,
        # no se intenta evadirlo.
        body_text = driver.find_element(By.TAG_NAME, 'body').text.lower()

        if (
            'unusual traffic' in body_text
            or 'tráfico inusual' in body_text
            or 'captcha' in body_text
        ):
            return {
                'Google_Name': '',
                'Google_Address': '',
                'Postal_Code': '',
                'Search_Query': query,
                'Search_Status': 'MANUAL REVIEW - CAPTCHA'
            }

        # Primero intentamos leer una ficha abierta directamente.
        address = get_address_from_maps(driver)

        # Si Maps devolvió una lista, abrimos el primer resultado.
        if address == 'Address not found':
            clicked = click_first_result_if_needed(driver)

            if clicked:
                address = get_address_from_maps(driver)

        google_name = get_google_name(driver)
        postal_code = extract_postal_code(address)

        if address == 'Address not found':
            status = 'REVIEW'
        elif postal_code == 'Postal code not found':
            status = 'ADDRESS FOUND - CP REVIEW'
        else:
            status = 'OK'

        print(f'  Nombre: {google_name}')
        print(f'  Dirección: {address}')
        print(f'  CP: {postal_code}')
        print(f'  Estado: {status}')

        # Pequeña pausa entre búsquedas para no saturar la página.
        time.sleep(1)

        return {
            'Google_Name': google_name,
            'Google_Address': address,
            'Postal_Code': postal_code,
            'Search_Query': query,
            'Search_Status': status
        }

    except Exception as e:
        print(f'Error con {query}: {e}')

        return {
            'Google_Name': '',
            'Google_Address': 'Error',
            'Postal_Code': '',
            'Search_Query': query,
            'Search_Status': 'ERROR'
        }


In [10]:
# Procesar todas las filas del Excel.
# Esta parte escala automáticamente al número de registros del archivo.
results = locations.apply(
    search_google_maps,
    axis=1
)

# Agregar los resultados al DataFrame original.
locations['Google_Name'] = results.apply(lambda x: x['Google_Name'])
locations['Google_Address'] = results.apply(lambda x: x['Google_Address'])
locations['Postal_Code'] = results.apply(lambda x: x['Postal_Code'])
locations['Search_Query'] = results.apply(lambda x: x['Search_Query'])
locations['Search_Status'] = results.apply(lambda x: x['Search_Status'])


Buscando: MX IMSS Hospital Centro Medico Siglo XXI
  Nombre: Centro Médico Nacional Siglo XXI
  Dirección: Av. Cuauhtémoc 330, Doctores, Cuauhtémoc, 06720 Ciudad de México, CDMX, México
  CP: 06720
  Estado: OK
Buscando: MX Ángeles Metropolitano Ciudad de México
  Nombre: Hospital Angeles Metropolitano
  Dirección: Tlacotalpan 59, Centro Urbano Pdte. Juárez, Roma Sur, Cuauhtémoc, 06760 Cuauhtémoc, CDMX
  CP: 06760
  Estado: OK
Buscando: MX Hospital de alta especialidad del Bajío Guanajuato
  Nombre: Hospital Regional de Alta Especialidad del Bajío
  Dirección: Blvd. Milenio 130, San Carlos la Roncha, 37544 León de los Aldama, Gto.
  CP: 37544
  Estado: OK
Buscando: AR Hospital General de Agudos José María Ramos Mejía Buenos Aires
  Nombre: Hospital General de Agudos José María Ramos Mejía
  Dirección: Gral. Urquiza 609, C1221ADC Cdad. Autónoma de Buenos Aires, Argentina
  CP: Postal code not found
  Estado: ADDRESS FOUND - CP REVIEW
Buscando: CO Hosítal Universitario San Antonio Bogotá

In [11]:
# Revisar resultados antes de exportar
locations[
    [
        'Nombre',
        'Google_Name',
        'Google_Address',
        'Postal_Code',
        'Search_Status'
    ]
].head(20)


,Nombre,Google_Name,Google_Address,Postal_Code,Search_Status
0,IMSS Hospital Centro Medico Siglo XXI,Centro Médico Nacional Siglo XXI,"Av. Cuauhtémoc 330, Doctores, Cuauhtémoc, 0672...",06720,OK
1,Ángeles Metropolitano,Hospital Angeles Metropolitano,"Tlacotalpan 59, Centro Urbano Pdte. Juárez, Ro...",06760,OK
2,Hospital de alta especialidad del Bajío,Hospital Regional de Alta Especialidad del Bajío,"Blvd. Milenio 130, San Carlos la Roncha, 37544...",37544,OK
3,Hospital General de Agudos José María Ramos Mejía,Hospital General de Agudos José María Ramos Mejía,"Gral. Urquiza 609, C1221ADC Cdad. Autónoma de ...",Postal code not found,ADDRESS FOUND - CP REVIEW
4,Hosítal Universitario San Antonio,Hospital Universitario Nacional de Colombia,"Única sede, Cl. 44 #59 -75, Teusaquillo, Bogot...",Postal code not found,ADDRESS FOUND - CP REVIEW


In [12]:
# Cerrar Chrome después de procesar todas las ubicaciones
driver.quit()

# Guardar el archivo enriquecido
locations.to_excel(
    'ArchivosExcel/DireccionesRevisadas.xlsx',
    index=False
)

print('Archivo guardado en: ArchivosExcel/DireccionesRevisadas.xlsx')


Archivo guardado en: ArchivosExcel/DireccionesRevisadas.xlsx
